# L2 (结构 + 成分特征, v2) — 按化学式分组的 5 折交叉验证

与官方折协议相比, **只改了数据怎么分**:
- 外层: `GroupKFold(5)`, 按 reduced formula 分组 (划分来自 `make_group_splits.py`, L1/L2 共用)
- 内层早停验证集: `GroupShuffleSplit(test_size=0.2, random_state=42)`, 同样来自共用文件

其余不变: 特征 = `matbench_cache` 里的官方 5 个测试折缓存 (不重新提取), 后处理 (去重列、one-hot、
bool→int、转数值、inf→NaN) 与原 `extract_features` 相同; 均值填补在**外层训练折**上计算 (与官方协议相同);
XGB 参数、SEED=42、上限 8000 轮、早停 200 轮、截断 均与主协议相同。
不调用 `task.record`, MAE/RMSE 自己算。SHAP、特征重要性图、GNN 导出 关闭。

In [1]:
# ==================== Cell 1: 导入 ====================
import json
import time
import hashlib
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import sklearn
from matbench.bench import MatbenchBenchmark
from matminer.featurizers.composition import ElementProperty, BandCenter
from matminer.featurizers.structure import (
    DensityFeatures,
    GlobalSymmetryFeatures,
    StructuralHeterogeneity,
    ChemicalOrdering,
    Dimensionality,
)
from matminer.featurizers.structure.sites import SiteStatsFingerprint
import xgboost as xgb

print('✅ 所有库导入成功!')

~/miniforge3/envs/matbench3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入成功!


In [2]:
# ==================== Cell 2: 配置 ====================
PROTOCOL    = "group"
CONFIG_NAME = "L2"
TASK_NAME   = "matbench_mp_gap"
SEED        = 42
EXPECTED_N  = 106_113

NUM_BOOST_ROUND       = 8000     # 与主协议相同
EARLY_STOPPING_ROUNDS = 200

CACHE_DIR = Path("../matbench_cache")       # 只读, 本 notebook 不会写入
SPLIT_DIR = Path("../groupcv_splits")       # make_group_splits.py 的输出
OUT_DIR   = Path("../matbench_outputs_v2_groupcv")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FOLDS_TO_RUN  = [0, 1, 2, 3, 4]
FORCE_RETRAIN = False            # 已有某折的预测和 json 就跳过训练, 直接读
N_VERIFY_FAST = 20               # 每个官方折: 快的 featurizer 抽多少条重算
N_VERIFY_SLOW = 4                # 每个官方折: 慢的 featurizer (CrystalNN / Voronoi) 抽多少条
ALLOW_MIXED_DTYPES = False       # 各官方折的非数值列不一致时默认停下 (见 Cell 4 的说明), 确认后才设 True

print(f'✅ 配置完成: {PROTOCOL} / {CONFIG_NAME} / seed {SEED} / 输出 {OUT_DIR}')

✅ 配置完成: group / L2 / seed 42 / 输出 ../matbench_outputs_v2_groupcv


In [3]:
# ==================== Cell 3: 加载数据和共用划分 ====================
mb   = MatbenchBenchmark(autoload=False)
task = next(t for t in mb.tasks if t.dataset_name == TASK_NAME)
task.load()
target_col = task.metadata["target"].strip()

df_all = task.df.rename(columns=lambda x: x.strip())
df_all.index = [str(i) for i in df_all.index]

folds = pd.read_csv(SPLIT_DIR / "folds.csv", dtype={"id": str, "formula": str}, float_precision="round_trip", keep_default_na=False)
inner = pd.read_csv(SPLIT_DIR / "inner_splits.csv", dtype={"id": str}, float_precision="round_trip", keep_default_na=False)
split_meta = json.loads((SPLIT_DIR / "split_meta.json").read_text(encoding="utf-8"))

assert len(folds) == EXPECTED_N and folds["id"].is_unique, "folds.csv 条数或 id 不对"
assert folds["id"].tolist() == df_all.index.tolist(), "folds.csv 的 id 顺序与 task.df 不一致"
assert np.array_equal(folds["y_true"].to_numpy(np.float64), df_all[target_col].to_numpy(np.float64)), "真值不一致"
_formula_now = [s.composition.reduced_formula for s in df_all["structure"]]
assert folds["formula"].tolist() == _formula_now, "folds.csv 的化学式与当前 pymatgen 计算结果不一致"

formula_by_id = folds.set_index("id")["formula"]
y_by_id       = folds.set_index("id")["y_true"]
print(f'✅ 数据 {len(df_all)} 条, 划分文件核对通过 (sklearn 生成时版本: {split_meta["versions"]["scikit-learn"]})')

2026-10-10 18:41:44 INFO     Initialized benchmark 'matbench_v0.1' with 13 tasks: 
['matbench_dielectric',
 'matbench_expt_gap',
 'matbench_expt_is_metal',
 'matbench_glass',
 'matbench_jdft2d',
 'matbench_log_gvrh',
 'matbench_log_kvrh',
 'matbench_mp_e_form',
 'matbench_mp_gap',
 'matbench_mp_is_metal',
 'matbench_perovskites',
 'matbench_phonons',
 'matbench_steels']
2026-10-10 18:41:44 INFO     Loading dataset 'matbench_mp_gap'...
2026-10-10 18:42:59 INFO     Dataset 'matbench_mp_gap loaded.
✅ 数据 106113 条, 划分文件核对通过 (sklearn 生成时版本: 1.6.1)


In [4]:
# ==================== Cell 4: 特征 — 读缓存 + 原 extract_features 的后处理 ====================
# featurizer 列表与原 extract_features 完全相同 (这里只用来: 确定缓存文件名, 以及抽样重算核对)
def make_featurizers():
    struct_featurizer_list = [
        ('DensityFeatures',         DensityFeatures()),
        ('GlobalSymmetryFeatures',  GlobalSymmetryFeatures()),
        ('StructuralHeterogeneity', StructuralHeterogeneity()),
        ('ChemicalOrdering',        ChemicalOrdering()),
        ('Dimensionality',          Dimensionality()),
        ('SiteStatsFingerprint',    SiteStatsFingerprint.from_preset('CrystalNNFingerprint_ops')),
    ]
    comp_featurizer_list = [
        ('ElementProperty', ElementProperty.from_preset('magpie')),
        ('BandCenter',      BandCenter()),
    ]
    return struct_featurizer_list, comp_featurizer_list

STRUCT_FZ, COMP_FZ = make_featurizers()
FZ_NAMES = [n for n, _ in STRUCT_FZ] + [n for n, _ in COMP_FZ]


def load_cached_parts(split_name, n_rows):
    """读一个 split 的 8 个缓存文件 (只读; 缺文件直接报错, 绝不重新提取)"""
    parts = {}
    for name in FZ_NAMES:
        p = CACHE_DIR / f'{split_name}_{name}.pkl'
        if not p.exists():
            raise FileNotFoundError(f'缺缓存 {p}; 本 notebook 不重新提取特征')
        part = pd.read_pickle(p)
        assert len(part) == n_rows, f'{p.name}: 行数 {len(part)} ≠ {n_rows}'
        parts[name] = part.reset_index(drop=True)
    return parts


def postprocess_features(X: pd.DataFrame) -> pd.DataFrame:
    """与原 extract_features 中 concat 之后的处理完全相同"""
    # 防止不同 featurizer 产生重名列
    X = X.loc[:, ~X.columns.duplicated()].copy()

    # 找出非数值列，比如 crystal_system 这种字符串类别
    non_numeric_cols = X.select_dtypes(exclude=[np.number, "bool"]).columns.tolist()
    if non_numeric_cols:
        print(f"  ⚠️ 发现非数值特征列，将进行 one-hot 编码: {non_numeric_cols}")
        X = pd.get_dummies(X, columns=non_numeric_cols, dummy_na=True)

    # 把 bool 转成 0/1
    bool_cols = X.select_dtypes(include=["bool"]).columns
    if len(bool_cols) > 0:
        X[bool_cols] = X[bool_cols].astype(int)

    # 全部强制转成数值，转不了的变 NaN
    X = X.apply(pd.to_numeric, errors="coerce")

    # 处理 inf；NaN 保留，填充延后到 fold 内用训练集统计量执行
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


# 官方 5 个测试折的缓存按行号存 (没有 id), 行顺序 = 当时 get_test_data(fold) 的顺序
raw_parts_by_fold, raw_by_fold, nonnum_by_fold = {}, {}, {}
for fold in task.folds:
    tdf  = task.get_test_data(fold, as_type="df").rename(columns=lambda x: x.strip())
    tids = [str(i) for i in tdf.index]
    exp  = folds[folds["official_fold"] == int(fold)].sort_values("official_pos")["id"].tolist()
    assert tids == exp, f"官方 fold {fold} 的 id 顺序与 folds.csv 记录不一致"
    parts = load_cached_parts(f"fold_{fold}_test", len(tids))
    for p in parts.values():
        p.index = tids
    raw = pd.concat(list(parts.values()), axis=1)
    raw = raw.loc[:, ~raw.columns.duplicated()]
    raw_parts_by_fold[int(fold)] = parts
    raw_by_fold[int(fold)] = raw
    nonnum_by_fold[int(fold)] = raw.select_dtypes(exclude=[np.number, "bool"]).columns.tolist()

cols0 = list(raw_by_fold[0].columns)
assert all(list(r.columns) == cols0 for r in raw_by_fold.values()), "各折缓存的原始特征列不一致"

# 诊断: 各官方折里"非数值列"是否一致。原协议是每个 split 单独 one-hot 的;
# 若某列在一些折是 bool、另一些折因提取失败变成 object, 原协议下两种编码会混用。这里如实报告。
print("  各官方测试折的非数值列:")
for f, cols in nonnum_by_fold.items():
    print(f"    fold {f}: {cols}")

# 同时看一眼官方"训练"折缓存里 GlobalSymmetryFeatures 的非数值列 (只读, 只是诊断原协议有没有受影响)
print("  官方训练折缓存 GlobalSymmetryFeatures 的非数值列 (诊断原协议用):")
for fold in task.folds:
    p = CACHE_DIR / f"fold_{fold}_train_GlobalSymmetryFeatures.pkl"
    if p.exists():
        print(f"    fold {fold} train: {pd.read_pickle(p).select_dtypes(exclude=[np.number, 'bool']).columns.tolist()}")

if len({tuple(c) for c in nonnum_by_fold.values()}) > 1:
    msg = ("各官方测试折的非数值列不一致 (见上)。通常是某折有提取失败的行 (NaN), 让 bool 列变成 object 被 one-hot。"
           "本 notebook 会在全表上统一 one-hot (所有行编码一致), 但这与原协议逐 split 编码的结果不同。"
           "请先确认, 再把 ALLOW_MIXED_DTYPES 设为 True 重跑。")
    if not ALLOW_MIXED_DTYPES:
        raise RuntimeError(msg)
    print("  ⚠️ " + msg)

# 先把 5 折原始特征拼成全表, 再统一做一次后处理 → 所有行同一套 one-hot 列, 不需要再补列
raw_all = pd.concat([raw_by_fold[f] for f in sorted(raw_by_fold)], axis=0).loc[folds["id"]]
X_all = postprocess_features(raw_all)
assert len(X_all) == EXPECTED_N and X_all.index.is_unique
n_nan = int(X_all.isna().sum().sum())
print(f'  特征表: {X_all.shape}, NaN 个数 {n_nan}')

# --- 核对 1 (全量): 缓存行与结构没有错位 ---
structs = df_all.loc[X_all.index, "structure"]
ep_all = pd.concat([raw_parts_by_fold[f]["ElementProperty"] for f in sorted(raw_parts_by_fold)]).loc[X_all.index]
zmin = structs.map(lambda s: min(e.Z for e in s.composition.elements)).to_numpy(np.float64)
zmax = structs.map(lambda s: max(e.Z for e in s.composition.elements)).to_numpy(np.float64)
assert np.array_equal(ep_all["MagpieData minimum Number"].to_numpy(np.float64), zmin), "ElementProperty 缓存错位 (min Z)"
assert np.array_equal(ep_all["MagpieData maximum Number"].to_numpy(np.float64), zmax), "ElementProperty 缓存错位 (max Z)"
den_all = pd.concat([raw_parts_by_fold[f]["DensityFeatures"] for f in sorted(raw_parts_by_fold)]).loc[X_all.index]
den_now = structs.map(lambda s: s.density).to_numpy(np.float64)
assert np.allclose(den_all["density"].to_numpy(np.float64), den_now, rtol=1e-6, atol=1e-9, equal_nan=True), \
    "DensityFeatures 缓存错位 (density)"
print('  ✅ 全量核对通过: ElementProperty 的 min/max 原子序数、DensityFeatures 的 density 都和对应结构一致')

# --- 核对 2 (抽样): 每个 featurizer 抽样重算, 与缓存比较 (报告用; 错位会表现为大量不一致) ---
SLOW = {"StructuralHeterogeneity", "ChemicalOrdering", "Dimensionality", "SiteStatsFingerprint"}

def _same(a, b):
    if a is None or (isinstance(a, float) and np.isnan(a)):
        return b is None or pd.isna(b)
    if b is None or (not isinstance(b, str) and pd.isna(b)):
        return False
    try:
        return bool(np.isclose(float(a), float(b), rtol=1e-4, atol=1e-6))
    except (TypeError, ValueError):
        return str(a) == str(b)

_rng = np.random.RandomState(SEED)
verify_report = {}
for name, fz in STRUCT_FZ + COMP_FZ:
    labels = fz.feature_labels()
    n_ok = n_chk = 0
    for f in sorted(raw_parts_by_fold):
        cached = raw_parts_by_fold[f][name]
        n_s = N_VERIFY_SLOW if name in SLOW else N_VERIFY_FAST
        for i in _rng.choice(cached.index.to_numpy(), min(n_s, len(cached)), replace=False):
            s = df_all.at[i, "structure"]
            obj = s.composition if name in ("ElementProperty", "BandCenter") else s
            try:
                vals = fz.featurize(obj)
            except Exception:
                vals = [np.nan] * len(labels)          # 与原来 ignore_errors=True 一致
            n_ok += all(_same(v, cached.at[i, l]) for v, l in zip(vals, labels))
            n_chk += 1
    verify_report[name] = (n_ok, n_chk)
    print(f'  抽样重算 {name:24s} {n_ok}/{n_chk} 一致')
bad = [n for n, (o, c) in verify_report.items() if o < 0.5 * c]
if bad:
    raise RuntimeError(f"抽样重算大量不一致, 缓存可能错位: {bad}")
if any(o < c for o, c in verify_report.values()):
    print('  ⚠️ 有少量不一致: 若差异很小, 多半是库版本差异而非错位; 请看具体条目后再决定')

FEATURE_COLUMNS = list(X_all.columns)
(OUT_DIR / "feature_columns.json").write_text(json.dumps(FEATURE_COLUMNS, ensure_ascii=False), encoding="utf-8")
del raw_parts_by_fold, raw_by_fold, raw_all

~/miniforge3/envs/matbench3.9/lib/python3.9/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)
~/miniforge3/envs/matbench3.9/lib/python3.9/site-packages/matminer/featurizers/composition/element.py:216: UserWarning: BandCenter(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by th

  各官方测试折的非数值列:
    fold 0: ['crystal_system']
    fold 1: ['crystal_system']
    fold 2: ['crystal_system']
    fold 3: ['crystal_system']
    fold 4: ['crystal_system']
  官方训练折缓存 GlobalSymmetryFeatures 的非数值列 (诊断原协议用):
    fold 0 train: ['crystal_system']
    fold 1 train: ['crystal_system']
    fold 2 train: ['crystal_system']
    fold 3 train: ['crystal_system']
    fold 4 train: ['crystal_system']
  ⚠️ 发现非数值特征列，将进行 one-hot 编码: ['crystal_system']
  特征表: (106113, 283), NaN 个数 94751
  ✅ 全量核对通过: ElementProperty 的 min/max 原子序数、DensityFeatures 的 density 都和对应结构一致
  抽样重算 DensityFeatures          100/100 一致
  抽样重算 GlobalSymmetryFeatures   100/100 一致
  抽样重算 StructuralHeterogeneity  20/20 一致
  抽样重算 ChemicalOrdering         20/20 一致


~/miniforge3/envs/matbench3.9/lib/python3.9/site-packages/pymatgen/analysis/local_env.py:4151: UserWarning: No oxidation states specified on sites! For better results, set the site oxidation states in the structure.
  warnings.warn(
~/miniforge3/envs/matbench3.9/lib/python3.9/site-packages/pymatgen/analysis/local_env.py:3944: UserWarning: CrystalNN: cannot locate an appropriate radius, covalent or atomic radii will be used, this can lead to non-optimal results.
  warnings.warn(


  抽样重算 Dimensionality           20/20 一致
  抽样重算 SiteStatsFingerprint     20/20 一致
  抽样重算 ElementProperty          100/100 一致
  抽样重算 BandCenter               100/100 一致


In [5]:
# ==================== Cell 5: XGBoost 训练函数 ====================
# 与 v2 原函数相同, 只有一处改动: 内层 80/20 不再用 train_test_split 随机切,
# 而是用共用文件里的分组划分 (fit_mask)。另外返回训练信息供写 json。
def xgb_train_and_predict(X_train, y_train, X_test, fold_idx: int, fit_mask):
    """训练 XGBoost（带早停）并预测测试集"""
    # [改动] 内层划分: 来自 inner_splits.csv (按化学式分组), 替代原来的
    #        train_test_split(X_train, y_train, test_size=0.2, random_state=SEED, shuffle=True)
    X_tr, X_val = X_train[fit_mask], X_train[~fit_mask]
    y_tr, y_val = y_train[fit_mask], y_train[~fit_mask]
    dtr   = xgb.DMatrix(X_tr,   label=y_tr)
    dval  = xgb.DMatrix(X_val,  label=y_val)
    dtest = xgb.DMatrix(X_test)

    params = {
        'objective':        'reg:squarederror',
        'eval_metric':      'mae',
        'eta':              0.03,
        'max_depth':        6,
        'min_child_weight': 5,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'lambda':           1.0,
        'alpha':            1e-3,
        'tree_method':      'hist',
        'seed':             SEED,
    }

    evals_result = {}
    print(f'  [XGBoost Fold {fold_idx}] 开始训练...')

    num_boost_round       = NUM_BOOST_ROUND
    early_stopping_rounds = EARLY_STOPPING_ROUNDS

    t0 = time.perf_counter()
    booster = xgb.train(
        params=params, dtrain=dtr,
        num_boost_round=num_boost_round,
        evals=[(dval, 'validation')],
        evals_result=evals_result,
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=50,
    )
    train_seconds = time.perf_counter() - t0

    best_it    = booster.best_iteration
    best_score = booster.best_score
    print(f'  [XGBoost Fold {fold_idx}] 完成: 最优轮数={best_it}, 验证MAE={best_score:.4f} eV')

    try:
        y_pred = booster.predict(dtest, iteration_range=(0, best_it + 1))
    except TypeError:
        y_pred = booster.predict(dtest, ntree_limit=best_it + 1)

    val_mae_list = evals_result['validation']['mae']
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(val_mae_list) + 1), val_mae_list, label='验证集 MAE', color='blue')
    plt.axvline(x=best_it + 1, color='red', linestyle='--', label=f'最优轮数: {best_it} (MAE={best_score:.4f})')
    plt.xlabel('训练轮数'); plt.ylabel('MAE (eV)')
    plt.title(f'XGBoost 训练曲线 - {PROTOCOL} fold {fold_idx}')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
    plt.close()

    info = {
        "best_iteration": int(best_it),
        "best_validation_mae": float(best_score),
        "n_rounds_run": int(len(val_mae_list)),
        "num_boost_round_max": int(num_boost_round),
        "early_stopping_rounds": int(early_stopping_rounds),
        "early_stopped": bool(len(val_mae_list) < num_boost_round),
        "train_seconds": round(train_seconds, 3),
        "params": params,
    }
    return booster, y_pred, info

In [6]:
# ==================== Cell 6: 主流程 (5 折) ====================
predictions_dir = OUT_DIR / "predictions_xgb"
predictions_dir.mkdir(parents=True, exist_ok=True)
VERSIONS = {"python": platform.python_version(), "xgboost": xgb.__version__,
            "scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__}

def ids_sha256(ids):
    return hashlib.sha256("\n".join(map(str, ids)).encode("utf-8")).hexdigest()

fold_rows = {}
for k in FOLDS_TO_RUN:
    print(f'\n{"=" * 60}\n[{CONFIG_NAME}] group fold {k}\n{"=" * 60}')
    test_ids  = folds.loc[folds["group_fold"] == k, "id"].to_numpy()
    train_ids = folds.loc[folds["group_fold"] != k, "id"].to_numpy()
    inn = inner[inner["outer_fold"] == k].set_index("id")["role"]
    assert len(inn) == len(train_ids) and set(inn.index) == set(train_ids), "内层划分与外层训练折不对应"
    fit_mask = (inn.loc[train_ids] == "fit").to_numpy()

    # 三项检查 (每折都再查一遍)
    f_tr, f_te = set(formula_by_id.loc[train_ids]), set(formula_by_id.loc[test_ids])
    f_fit, f_val = set(formula_by_id.loc[train_ids[fit_mask]]), set(formula_by_id.loc[train_ids[~fit_mask]])
    assert not (f_tr & f_te), f"fold {k}: 外层训练/测试化学式有交集"
    assert not (f_fit & f_val), f"fold {k}: 内层 fit/val 化学式有交集"

    raw_file  = predictions_dir / f"pred_raw_fold_{k}.npy"
    clip_file = predictions_dir / f"pred_fold_{k}.npy"
    info_file = OUT_DIR / f"xgb_training_info_{PROTOCOL}_{CONFIG_NAME}_seed{SEED}_fold_{k}.json"
    y_test = y_by_id.loc[test_ids].to_numpy(np.float64)
    split_fp = {"test_ids_sha256": ids_sha256(test_ids), "fit_ids_sha256": ids_sha256(train_ids[fit_mask]),
                "val_ids_sha256": ids_sha256(train_ids[~fit_mask])}

    if raw_file.exists() and clip_file.exists() and info_file.exists() and not FORCE_RETRAIN:
        print('  已有结果, 跳过训练 (先核对它是不是用当前划分和设置跑出来的)')
        old = json.loads(info_file.read_text(encoding="utf-8"))
        for key, val in {**split_fp, "protocol": PROTOCOL, "config": CONFIG_NAME, "seed": SEED,
                         "num_boost_round_max": NUM_BOOST_ROUND,
                         "early_stopping_rounds": EARLY_STOPPING_ROUNDS}.items():
            if old.get(key) != val:
                raise RuntimeError(f"fold {k}: 已有结果的 {key} 与当前不一致 ({old.get(key)} vs {val}); "
                                   f"请换输出目录或设 FORCE_RETRAIN = True")
        y_raw  = np.load(raw_file)
        y_clip = np.load(clip_file)
    else:
        X_train = X_all.loc[train_ids]
        X_test  = X_all.loc[test_ids]
        y_train = y_by_id.loc[train_ids].to_numpy()

        # 原协议这里有 align; 全表统一 one-hot 后两边列已完全相同, 保留这一步只作确认
        X_train, X_test = X_train.align(X_test, join="outer", axis=1, fill_value=0)
        assert list(X_train.columns) == FEATURE_COLUMNS

        # 填充: 统计量只从外层训练折来 (fit + val, 与官方折协议相同)
        col_means = X_train.mean(numeric_only=True)
        X_train   = X_train.fillna(col_means).fillna(0)
        X_test    = X_test.fillna(col_means).fillna(0)

        booster, y_raw, tinfo = xgb_train_and_predict(X_train, y_train, X_test, k, fit_mask)

        np.save(raw_file, y_raw)                 # clip 前的原始预测
        y_clip = np.maximum(y_raw, 0)            # 物理约束: 带隙非负
        np.save(clip_file, y_clip)

        err = y_clip.astype(np.float64) - y_test
        info = {
            "protocol": PROTOCOL, "config": CONFIG_NAME, "seed": SEED, "fold": int(k),
            **{key: tinfo[key] for key in ["best_iteration", "best_validation_mae", "n_rounds_run",
                                           "num_boost_round_max", "early_stopping_rounds",
                                           "early_stopped", "train_seconds"]},
            "MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err ** 2))),
            "n_raw_negative": int((y_raw < 0).sum()),
            "n_train": int(len(train_ids)), "n_test": int(len(test_ids)),
            "n_fit": int(fit_mask.sum()), "n_val": int((~fit_mask).sum()),
            "val_fraction_of_outer_train": round(float((~fit_mask).mean()), 6),
            "n_groups_train": len(f_tr), "n_groups_test": len(f_te),
            "n_groups_fit": len(f_fit), "n_groups_val": len(f_val),
            "formula_overlap_train_test": 0, "formula_overlap_fit_val": 0,
            **split_fp,
            "impute_scope": "outer training fold (fit + val), same as official-fold protocol",
            "n_features": int(X_train.shape[1]),
            "params": tinfo["params"], "versions": VERSIONS,
        }
        info_file.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f'  → MAE={info["MAE"]:.6f}  RMSE={info["RMSE"]:.6f}  best_it={info["best_iteration"]}  '
              f'early_stopped={info["early_stopped"]}')

    assert len(y_raw) == len(test_ids) and np.array_equal(np.maximum(y_raw, 0), y_clip)
    fold_rows[k] = pd.DataFrame({
        "protocol": PROTOCOL, "config": CONFIG_NAME, "seed": SEED, "fold": k, "id": test_ids,
        "formula": formula_by_id.loc[test_ids].to_numpy(),
        "formula_count_global": folds.set_index("id").loc[test_ids, "formula_count_global"].to_numpy(),
        "y_true": y_test, "y_raw": y_raw.astype(np.float64), "y_clip": y_clip.astype(np.float64),
    })


[L2] group fold 0
  [XGBoost Fold 0] 开始训练...
[0]	validation-mae:1.31955
[50]	validation-mae:0.73513
[100]	validation-mae:0.57145
[150]	validation-mae:0.51926
[200]	validation-mae:0.49111
[250]	validation-mae:0.47328
[300]	validation-mae:0.46153
[350]	validation-mae:0.45140
[400]	validation-mae:0.44344
[450]	validation-mae:0.43704
[500]	validation-mae:0.43228
[550]	validation-mae:0.42762
[600]	validation-mae:0.42293
[650]	validation-mae:0.41912
[700]	validation-mae:0.41595
[750]	validation-mae:0.41235
[800]	validation-mae:0.40961
[850]	validation-mae:0.40697
[900]	validation-mae:0.40437
[950]	validation-mae:0.40237
[1000]	validation-mae:0.40037
[1050]	validation-mae:0.39899
[1100]	validation-mae:0.39732
[1150]	validation-mae:0.39546
[1200]	validation-mae:0.39393
[1250]	validation-mae:0.39252
[1300]	validation-mae:0.39094
[1350]	validation-mae:0.38978
[1400]	validation-mae:0.38857
[1450]	validation-mae:0.38737
[1500]	validation-mae:0.38643
[1550]	validation-mae:0.38565
[1600]	validation

/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) Deja

  → MAE=0.313326  RMSE=0.543216  best_it=7973  early_stopped=False

[L2] group fold 1
  [XGBoost Fold 1] 开始训练...
[0]	validation-mae:1.29531
[50]	validation-mae:0.71033
[100]	validation-mae:0.55135
[150]	validation-mae:0.49678
[200]	validation-mae:0.47153
[250]	validation-mae:0.45432
[300]	validation-mae:0.44243
[350]	validation-mae:0.43398
[400]	validation-mae:0.42666
[450]	validation-mae:0.42025
[500]	validation-mae:0.41546
[550]	validation-mae:0.41088
[600]	validation-mae:0.40668
[650]	validation-mae:0.40316
[700]	validation-mae:0.40005
[750]	validation-mae:0.39734
[800]	validation-mae:0.39480
[850]	validation-mae:0.39226
[900]	validation-mae:0.39014
[950]	validation-mae:0.38789
[1000]	validation-mae:0.38611
[1050]	validation-mae:0.38445
[1100]	validation-mae:0.38272
[1150]	validation-mae:0.38107
[1200]	validation-mae:0.37936
[1250]	validation-mae:0.37769
[1300]	validation-mae:0.37641
[1350]	validation-mae:0.37489
[1400]	validation-mae:0.37345
[1450]	validation-mae:0.37200
[1500]	val

/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) Deja

  → MAE=0.342552  RMSE=0.601582  best_it=7995  early_stopped=False

[L2] group fold 2
  [XGBoost Fold 2] 开始训练...
[0]	validation-mae:1.30659
[50]	validation-mae:0.72214
[100]	validation-mae:0.56419
[150]	validation-mae:0.51039
[200]	validation-mae:0.48530
[250]	validation-mae:0.46852
[300]	validation-mae:0.45538
[350]	validation-mae:0.44644
[400]	validation-mae:0.43901
[450]	validation-mae:0.43221
[500]	validation-mae:0.42607
[550]	validation-mae:0.42109
[600]	validation-mae:0.41679
[650]	validation-mae:0.41229
[700]	validation-mae:0.40898
[750]	validation-mae:0.40592
[800]	validation-mae:0.40302
[850]	validation-mae:0.39999
[900]	validation-mae:0.39730
[950]	validation-mae:0.39496
[1000]	validation-mae:0.39278
[1050]	validation-mae:0.39075
[1100]	validation-mae:0.38879
[1150]	validation-mae:0.38714
[1200]	validation-mae:0.38545
[1250]	validation-mae:0.38394
[1300]	validation-mae:0.38249
[1350]	validation-mae:0.38122
[1400]	validation-mae:0.38006
[1450]	validation-mae:0.37890
[1500]	val

/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) Deja

  → MAE=0.326938  RMSE=0.544750  best_it=7999  early_stopped=False

[L2] group fold 3
  [XGBoost Fold 3] 开始训练...
[0]	validation-mae:1.29825
[50]	validation-mae:0.71265
[100]	validation-mae:0.55176
[150]	validation-mae:0.49631
[200]	validation-mae:0.46945
[250]	validation-mae:0.45377
[300]	validation-mae:0.44146
[350]	validation-mae:0.43200
[400]	validation-mae:0.42409
[450]	validation-mae:0.41838
[500]	validation-mae:0.41244
[550]	validation-mae:0.40738
[600]	validation-mae:0.40307
[650]	validation-mae:0.39951
[700]	validation-mae:0.39605
[750]	validation-mae:0.39294
[800]	validation-mae:0.39032
[850]	validation-mae:0.38803
[900]	validation-mae:0.38613
[950]	validation-mae:0.38411
[1000]	validation-mae:0.38178
[1050]	validation-mae:0.37976
[1100]	validation-mae:0.37778
[1150]	validation-mae:0.37595
[1200]	validation-mae:0.37448
[1250]	validation-mae:0.37310
[1300]	validation-mae:0.37150
[1350]	validation-mae:0.37033
[1400]	validation-mae:0.36892
[1450]	validation-mae:0.36787
[1500]	val

/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) Deja

  → MAE=0.330643  RMSE=0.557091  best_it=7999  early_stopped=False

[L2] group fold 4
  [XGBoost Fold 4] 开始训练...
[0]	validation-mae:1.29569
[50]	validation-mae:0.71213
[100]	validation-mae:0.55537
[150]	validation-mae:0.50321
[200]	validation-mae:0.47821
[250]	validation-mae:0.46168
[300]	validation-mae:0.45079
[350]	validation-mae:0.44292
[400]	validation-mae:0.43610
[450]	validation-mae:0.43043
[500]	validation-mae:0.42533
[550]	validation-mae:0.42007
[600]	validation-mae:0.41567
[650]	validation-mae:0.41185
[700]	validation-mae:0.40865
[750]	validation-mae:0.40586
[800]	validation-mae:0.40246
[850]	validation-mae:0.40021
[900]	validation-mae:0.39783
[950]	validation-mae:0.39556
[1000]	validation-mae:0.39368
[1050]	validation-mae:0.39222
[1100]	validation-mae:0.39044
[1150]	validation-mae:0.38886
[1200]	validation-mae:0.38733
[1250]	validation-mae:0.38600
[1300]	validation-mae:0.38463
[1350]	validation-mae:0.38357
[1400]	validation-mae:0.38250
[1450]	validation-mae:0.38137
[1500]	val

/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT_DIR / f'xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
/var/folders/9j/x4s4qppn2fb8h1c8fzflvkhr0000gn/T/ipykernel_42138/2369865367.py:61: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) Deja

  → MAE=0.333053  RMSE=0.562329  best_it=7988  early_stopped=False


In [7]:
# ==================== Cell 7: 汇总 ====================
res = pd.concat([fold_rows[k] for k in sorted(fold_rows)], ignore_index=True)
per_fold = res.assign(ae=(res.y_clip - res.y_true).abs(), se=(res.y_clip - res.y_true) ** 2) \
              .groupby("fold").agg(n=("id", "size"), MAE=("ae", "mean"), RMSE=("se", "mean"))
per_fold["RMSE"] = np.sqrt(per_fold["RMSE"])
with pd.option_context("display.float_format", "{:.6f}".format):
    print(per_fold)
    print(f'\n五折平均 MAE = {per_fold.MAE.mean():.6f}, RMSE = {per_fold.RMSE.mean():.6f}')

if sorted(fold_rows) == [0, 1, 2, 3, 4]:
    assert len(res) == EXPECTED_N and res["id"].is_unique, "汇总后条数不是 106,113 或有重复 id"
    out_csv = OUT_DIR / f"pred_rows_{PROTOCOL}_{CONFIG_NAME}_seed{SEED}.csv"
    res.to_csv(out_csv, index=False)
    print(f'\n✅ 逐条预测已保存: {out_csv}  ({len(res)} 行)')
else:
    print(f'\n⚠️ 只跑了 fold {sorted(fold_rows)}, 不写逐条预测汇总文件')

          n      MAE     RMSE
fold                         
0     21223 0.313326 0.543216
1     21223 0.342552 0.601582
2     21223 0.326938 0.544750
3     21222 0.330643 0.557091
4     21222 0.333053 0.562329

五折平均 MAE = 0.329303, RMSE = 0.561794

✅ 逐条预测已保存: ../matbench_outputs_v2_groupcv/pred_rows_group_L2_seed42.csv  (106113 行)


In [8]:
import json, hashlib
import pandas as pd
from pathlib import Path

SPLIT = Path("../groupcv_splits")
DIRS  = {"L1": Path("../outputs_v1_groupcv"), "L2": Path("../matbench_outputs_v2_groupcv")}
rd = dict(float_precision="round_trip", keep_default_na=False)
folds = pd.read_csv(SPLIT / "folds.csv", dtype={"id": str, "formula": str}, **rd)
inner = pd.read_csv(SPLIT / "inner_splits.csv", dtype={"id": str}, **rd)
form  = folds.set_index("id")["formula"]
sha   = lambda ids: hashlib.sha256("\n".join(ids).encode("utf-8")).hexdigest()

rows = []
for k in range(5):
    train = folds.loc[folds.group_fold != k, "id"].to_numpy()       # 顺序与 notebook 相同
    test  = folds.loc[folds.group_fold == k, "id"].to_numpy()
    inn   = inner[inner.outer_fold == k].set_index("id")["role"]
    assert len(inn) == len(train) and set(inn.index) == set(train), f"fold {k}: 内层文件与外层训练折不对应"
    mask  = (inn.loc[train] == "fit").to_numpy()
    fit, val = train[mask], train[~mask]
    fp = {"test": sha(test), "fit": sha(fit), "val": sha(val)}
    row = {"fold": k, "n_fit": len(fit), "n_val": len(val),
           "val_frac": round(len(val) / len(train), 6),
           "overlap_fit_val": len(set(form[fit]) & set(form[val])),          # 现算的交集数
           "overlap_train_test": len(set(form[train]) & set(form[test]))}
    for cfg, d in DIRS.items():
        j = json.loads((d / f"xgb_training_info_group_{cfg}_seed42_fold_{k}.json").read_text(encoding="utf-8"))
        for part in ("test", "fit", "val"):
            row[f"{cfg}_{part}_match"] = j[f"{part}_ids_sha256"] == fp[part]
    rows.append(row)

audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
ok = (audit.overlap_fit_val.eq(0).all() and audit.overlap_train_test.eq(0).all()
      and audit.filter(like="_match").all().all())
print("\n全部通过 ✅" if ok else "\n有不通过的项 ❌")
audit.to_csv(SPLIT / "split_audit.csv", index=False)
print(f"已保存: {SPLIT / 'split_audit.csv'}")

 fold  n_fit  n_val  val_frac  overlap_fit_val  overlap_train_test  L1_test_match  L1_fit_match  L1_val_match  L2_test_match  L2_fit_match  L2_val_match
    0  67322  17568  0.206950                0                   0           True          True          True           True          True          True
    1  67865  17025  0.200554                0                   0           True          True          True           True          True          True
    2  68591  16299  0.192001                0                   0           True          True          True           True          True          True
    3  67467  17424  0.205251                0                   0           True          True          True           True          True          True
    4  67678  17213  0.202766                0                   0           True          True          True           True          True          True

全部通过 ✅
已保存: ../groupcv_splits/split_audit.csv


In [9]:
import numpy as np
for name in ["test_preds.npz", "test_preds_clipped.npz"]:
    d = np.load(f"../results_v4/fold_0/{name}", allow_pickle=True)
    print(name, {k: (d[k].shape, d[k].dtype) for k in d.files})
    for k in d.files:
        print("  ", k, d[k][:3])

test_preds.npz {'ids': ((21223,), dtype('<U16')), 'preds': ((21223,), dtype('float32'))}
   ids ['mb-mp-gap-000003' 'mb-mp-gap-000004' 'mb-mp-gap-000011']
   preds [-0.01082987  0.6159152   0.00100218]
test_preds_clipped.npz {'ids': ((21223,), dtype('<U16')), 'preds': ((21223,), dtype('float64'))}
   ids ['mb-mp-gap-000003' 'mb-mp-gap-000004' 'mb-mp-gap-000011']
   preds [0.         0.61591518 0.00100218]
